In [5]:
import os
import numpy as np
import pandas as pd
import cv2

import splitfolders
import h5py
from matplotlib import pyplot as plt
%matplotlib inline
from matplotlib import rcParams
import seaborn as sns
from PIL import Image
import imutils 

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.applications import (
    InceptionResNetV2,
    ResNet50,
    InceptionV3,
    DenseNet121,
)
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.imagenet_utils import preprocess_input



In [2]:
train_set = 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train'
val_set = 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/val'
test_set = 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/test'


In [3]:
def init_data(train_dir: str, valid_dir: str) -> tuple:
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    
    train_data = train_datagen.flow_from_directory(
        directory=train_dir,
        class_mode='categorical',
        batch_size=64,
        seed=42,
        shuffle=False,
    )
    valid_data = valid_datagen.flow_from_directory(
        directory=valid_dir,
        class_mode='categorical',
        batch_size=64,
        seed=42,
        shuffle=False,
    )
    
    return train_data, valid_data

In [4]:
train_data, valid_data = init_data(train_dir=train_set, valid_dir=val_set)

Found 2144 images belonging to 3 classes.
Found 458 images belonging to 3 classes.


In [8]:
def build_transfer_learning_model(base_model):
    # `base_model` stands for the pretrained model
    # We want to use the learned weights, and to do so we must freeze them
    for layer in base_model.layers:
        layer.trainable = False
        
    # Declare a sequential model that combines the base model with custom layers
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=3, activation='softmax')
    ])
    
    # Compile the model
    model.compile(
        loss='categorical_crossentropy',
        optimizer='Adam',
        metrics=['accuracy']
    )
    
    return model

In [9]:
vgg_model = build_transfer_learning_model(
    base_model=VGG16(include_top=False, input_shape=(256, 256, 3), weights='imagenet')
)

58900480/58889256 [==============================] - 106s 2us/step


In [10]:
# Train the model for 10 epochs
vgg_hist = vgg_model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10
)

Epoch 1/10
 2/34 [>.............................] - ETA: 16:23 - loss: 1.3812 - accuracy: 0.3516

KeyboardInterrupt: 

In [23]:
# IMAGE_SIZE = 256

# model_dir ="C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Brain MRI images Classification/Models/RadImageNet-ResNet50_notop.h5"
# # base_model = ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, pooling="avg")
# base_model = ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False, pooling="avg")
# base_model.summary()

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_10 (InputLayer)          [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 262, 262, 3)  0           ['input_10[0][0]']               
                                                                                                  
 conv1_conv (Conv2D)            (None, 128, 128, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

In [19]:
# base_model_config = base_model.get_config() 

In [21]:
# channel_num = 1
# base_model_config["layers"][0]["config"]["batch_input_shape"] =(None, IMAGE_SIZE, IMAGE_SIZE, channel_num)


In [22]:
# updated_base_model = Model.from_config(base_model_config)
# print(updated_base_model.summary())

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_9 (InputLayer)           [(None, 256, 256, 1  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 262, 262, 1)  0           ['input_9[0][0]']                
                                                                                                  
 conv1_conv (Conv2D)            (None, 128, 128, 64  3200        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

In [ ]:

# data = []
# Home = 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train'
# for folder in sorted(os.listdir('C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train')):
#     for file in sorted(os.listdir('C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/'+folder)):
#         file_path = Home + '/' + folder + '/' + file

#         file_name = file.split('.')[0]
#         data.append([file_name, folder, file_path])

# train_set = pd.DataFrame(data, columns=['File_Name', 'Folder', 'File_Path'])
# print(train_set)
# train_set.to_csv('Train_set.csv')
